In [1]:
# Cell 1: Import các thư viện cần thiết
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
import pickle
from pathlib import Path

print("✓ Import thành công các thư viện")

✓ Import thành công các thư viện


d:\code\TTCS\PROJECT_ThucTapCoSo\AI_Services\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: Thiết lập đường dẫn và kiểm tra file tồn tại
# Đường dẫn lùi ra 1 cấp để vào thư mục BànGiao
file_path = '../BànGiao/lastfm_clean.csv'

# Kiểm tra file có tồn tại không
if not Path(file_path).exists():
    raise FileNotFoundError(f"Không tìm thấy file: {file_path}")

print(f"✓ File tồn tại: {file_path}")

✓ File tồn tại: ../BànGiao/lastfm_clean.csv


In [4]:
# Cell 3: Nạp dữ liệu và xem tổng quan
print(f"Đang nạp dữ liệu từ: {file_path}...")
df = pd.read_csv(file_path)

print(f"\n✓ Load thành công {len(df):,} dòng dữ liệu")
print(f"✓ Số user: {df['userID'].nunique():,}")
print(f"✓ Số artist: {df['artistID'].nunique():,}")
print(f"\nThống kê rating:")
print(df['rating'].describe())

# Xem 5 dòng đầu để xác nhận cấu trúc
print("\nDữ liệu mẫu:")
df.head()

Đang nạp dữ liệu từ: ../BànGiao/lastfm_clean.csv...

✓ Load thành công 92,834 dòng dữ liệu
✓ Số user: 1,892
✓ Số artist: 17,632

Thống kê rating:
count    92834.000000
mean         2.996456
std          1.416694
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64

Dữ liệu mẫu:


,userID,artistID,rating
0,2,51,5
1,2,52,5
2,2,53,5
3,2,54,5
4,2,55,5


In [5]:
# Cell 4: Chuyển đổi userID và artistID sang category
print("Đang chuyển đổi userID và artistID sang category...")

df['userID'] = df['userID'].astype("category")
df['artistID'] = df['artistID'].astype("category")

# Tạo mapping để tra cứu (Quan trọng để test model sau này)
user_map = dict(enumerate(df['userID'].cat.categories))
artist_map = dict(enumerate(df['artistID'].cat.categories))

# Tạo reverse mapping để tìm index từ ID gốc
user_to_idx = {v: k for k, v in user_map.items()}
artist_to_idx = {v: k for k, v in artist_map.items()}

print(f"✓ Đã tạo mapping cho {len(user_map):,} users và {len(artist_map):,} artists")

Đang chuyển đổi userID và artistID sang category...
✓ Đã tạo mapping cho 1,892 users và 17,632 artists


In [6]:
# Cell 5: Tạo ma trận thưa (Artist × User) 
print("Đang tạo sparse matrix (Artist × User)...")

# Dùng đúng tên cột userID, artistID và rating từ file thực tế của Diti
item_user_matrix = sparse.csr_matrix(
    (df['rating'].astype(float), 
     (df['artistID'].cat.codes, 
      df['userID'].cat.codes))
)

# Tính toán các chỉ số chuyên sâu cho báo cáo
n_users = item_user_matrix.shape[1]
n_items = item_user_matrix.shape[0]
n_interactions = item_user_matrix.nnz
possible_interactions = n_users * n_items
sparsity = 100 * (1 - n_interactions / possible_interactions)
memory_usage = item_user_matrix.data.nbytes / (1024**2)

print(f"✓ Ma trận sparse đã tạo thành công:")
print(f"  - Hình dáng (Shape): {item_user_matrix.shape} (Artists × Users)")
print(f"  - Số tương tác thực tế: {n_interactions:,}")
print(f"  - Độ thưa (Sparsity): {sparsity:.2f}%")
print(f"  - Chiếm dụng RAM: {memory_usage:.2f} MB")

Đang tạo sparse matrix (Artist × User)...
✓ Ma trận sparse đã tạo thành công:
  - Hình dáng (Shape): (17632, 1892) (Artists × Users)
  - Số tương tác thực tế: 92,834
  - Độ thưa (Sparsity): 99.72%
  - Chiếm dụng RAM: 0.71 MB


In [7]:
# Cell 6: Khởi tạo và huấn luyện mô hình ALS
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ALS")
print("=" * 60)

# Cấu hình model
model = implicit.als.AlternatingLeastSquares(
    factors=50,           # Số lượng đặc trưng ẩn
    iterations=20,        # Số vòng lặp tối ưu
    regularization=0.1,   # Tránh học vẹt (overfitting)
    random_state=42,      # Đảm bảo kết quả giống nhau mỗi lần chạy
    use_gpu=False         # Chạy trên CPU để ổn định môi trường venv
)

# QUAN TRỌNG: Phải dùng .T để chuyển ma trận về dạng (Users x Artists)
# Chúng ta ép kiểu tocsr() để đảm bảo tốc độ tính toán nhanh nhất
print("Đang huấn luyện (có thể mất vài giây)...")
model.fit(item_user_matrix.T.tocsr(), show_progress=True)

print("\n✓ Huấn luyện hoàn tất!")

d:\code\TTCS\PROJECT_ThucTapCoSo\AI_Services\venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH ALS
Đang huấn luyện (có thể mất vài giây)...


100%|██████████| 20/20 [00:00<00:00, 27.39it/s]


✓ Huấn luyện hoàn tất!


In [12]:
# Cell 7: Kiểm tra nhanh model có hoạt động không (Bản sửa lỗi Unpack)
print("KIỂM TRA NHANH MODEL")
print("=" * 60)

# 1. Chọn user đầu tiên để test
test_user_idx = 2
test_user_id = user_map[test_user_idx]

print(f"Đang tạo gợi ý cho User ID gốc: {test_user_id} (Vị trí ma trận: {test_user_idx})")

# Lấy ma trận User-Artist đầy đủ
full_user_items = item_user_matrix.T.tocsr()

# SỬA LỖI: Bản mới trả về 2 mảng (ids, scores) riêng biệt
ids, scores = model.recommend(
    userid=test_user_idx,
    user_items=full_user_items[test_user_idx], 
    N=10,
    filter_already_liked_items=True
)

print(f"\nTop 10 nghệ sĩ gợi ý (Dựa trên mô hình Baseline MF):")
# Dùng zip để ghép cặp ID và Score lại với nhau
for i, (artist_idx, score) in enumerate(zip(ids, scores), 1):
    artist_id = artist_map[artist_idx]
    print(f"  {i}. Artist ID: {artist_id:<10} | Độ ưu tiên (Score): {score:.4f}")

print("-" * 60)

# 2. Tương tự cho phần tìm nghệ sĩ tương tự
test_artist_idx = 0
test_artist_id = artist_map[test_artist_idx]

# SỬA LỖI: similar_items cũng trả về 2 mảng riêng biệt
sim_ids, sim_scores = model.similar_items(
    itemid=test_artist_idx,
    N=5
)

print(f"Top 5 nghệ sĩ tương tự với Artist ID {test_artist_id}:")
for i, (artist_idx, score) in enumerate(zip(sim_ids, sim_scores), 1):
    artist_id = artist_map[artist_idx]
    print(f"  - Artist ID: {artist_id:<10} | Độ tương đồng: {score:.4f}")

KIỂM TRA NHANH MODEL
Đang tạo gợi ý cho User ID gốc: 4 (Vị trí ma trận: 2)

Top 10 nghệ sĩ gợi ý (Dựa trên mô hình Baseline MF):
  1. Artist ID: 511        | Độ ưu tiên (Score): 0.8661
  2. Artist ID: 1001       | Độ ưu tiên (Score): 0.8014
  3. Artist ID: 221        | Độ ưu tiên (Score): 0.7127
  4. Artist ID: 486        | Độ ưu tiên (Score): 0.6726
  5. Artist ID: 238        | Độ ưu tiên (Score): 0.6651
  6. Artist ID: 506        | Độ ưu tiên (Score): 0.6474
  7. Artist ID: 63         | Độ ưu tiên (Score): 0.6408
  8. Artist ID: 67         | Độ ưu tiên (Score): 0.6124
  9. Artist ID: 959        | Độ ưu tiên (Score): 0.6095
  10. Artist ID: 1713       | Độ ưu tiên (Score): 0.5563
------------------------------------------------------------
Top 5 nghệ sĩ tương tự với Artist ID 1:
  - Artist ID: 1          | Độ tương đồng: 1.0000
  - Artist ID: 397        | Độ tương đồng: 0.8896
  - Artist ID: 1157       | Độ tương đồng: 0.8673
  - Artist ID: 390        | Độ tương đồng: 0.8665
  - Artis

In [13]:
# Cell 8: Lưu model và metadata vào file pickle
import pickle
from pathlib import Path

# Tên file xuất ra
output_path = 'baseline_mf.pkl'

print(f"Đang đóng gói và lưu model vào {output_path}...")

# Gom tất cả các "mảnh ghép" quan trọng vào một Dictionary
# Việc này giúp khi load lại file ở Backend, bạn có đầy đủ công cụ để gợi ý ngay
model_data = {
    'model': model,
    'user_map': user_map,
    'artist_map': artist_map,
    'user_to_idx': user_to_idx,
    'artist_to_idx': artist_to_idx,
    'matrix_shape': item_user_matrix.shape,
    'config': {
        'factors': 50,
        'iterations': 20,
        'regularization': 0.1,
        'random_state': 42
    }
}

# Tiến hành ghi file dưới dạng Byte (wb)
with open(output_path, 'wb') as f:
    pickle.dump(model_data, f)

# Kiểm tra lại kích thước file để đảm bảo việc lưu thành công
file_size = Path(output_path).stat().st_size / (1024**2)
print(f"✓ Đã lưu thành công vào {output_path}")
print(f"✓ Kích thước file: {file_size:.2f} MB")

Đang đóng gói và lưu model vào baseline_mf.pkl...
✓ Đã lưu thành công vào baseline_mf.pkl
✓ Kích thước file: 4.01 MB


In [14]:
# Cell 9: Kiểm tra load lại model từ file (Đã sửa lỗi Unpack và Slicing)
print("KIỂM TRA NGHIỆM THU MODEL")
print("=" * 60)

# 1. Load toàn bộ dữ liệu từ file pickle
with open('baseline_mf.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

# Giải nén các thành phần
loaded_model = loaded_data['model']
loaded_user_map = loaded_data['user_map']
loaded_artist_map = loaded_data['artist_map']

print("✓ Load model và Metadata thành công!")
print(f"✓ Quy mô: {len(loaded_user_map):,} Users | {len(loaded_artist_map):,} Artists")
print(f"✓ Cấu hình đã dùng: {loaded_data['config']}")

# 2. Test lại logic gợi ý (Giả lập Backend gọi hàm)
test_idx = 0
# Vẫn dùng item_user_matrix từ RAM để đối chiếu
user_vector = item_user_matrix.T.tocsr()[test_idx]

# Sửa lỗi: Hứng 2 mảng (ids, scores) và truyền đúng 1 hàng dữ liệu
ids, scores = loaded_model.recommend(
    userid=test_idx,
    user_items=user_vector,
    N=5,
    filter_already_liked_items=True
)

print(f"\nKết quả gợi ý từ file đã lưu (User {loaded_user_map[test_idx]}):")
for i, (artist_idx, score) in enumerate(zip(ids, scores), 1):
    print(f"  {i}. Artist {loaded_artist_map[artist_idx]:<10} | Score: {score:.4f}")

print("\n✓ XÁC NHẬN: Model hoạt động hoàn hảo sau khi đóng gói!")

KIỂM TRA NGHIỆM THU MODEL
✓ Load model và Metadata thành công!
✓ Quy mô: 1,892 Users | 17,632 Artists
✓ Cấu hình đã dùng: {'factors': 50, 'iterations': 20, 'regularization': 0.1, 'random_state': 42}

Kết quả gợi ý từ file đã lưu (User 2):
  1. Artist 1001       | Score: 1.1579
  2. Artist 187        | Score: 0.9782
  3. Artist 238        | Score: 0.8270
  4. Artist 1014       | Score: 0.8146
  5. Artist 193        | Score: 0.7920

✓ XÁC NHẬN: Model hoạt động hoàn hảo sau khi đóng gói!


In [15]:
# Cell 10: Đánh giá độ phủ (Coverage) - Chỉ số quan trọng cho báo cáo
print("ĐÁNH GIÁ CHẤT LƯỢNG BASELINE MODEL")
print("=" * 60)

# Coverage: Tỉ lệ nghệ sĩ được hệ thống "chạm" tới trong số 17,632 nghệ sĩ
recommended_artists = set()
sample_size = min(200, len(user_map))  # Tăng lên 200 users để kết quả khách quan hơn

print(f"Đang tính toán Coverage dựa trên {sample_size} người dùng mẫu...")

# Lấy ma trận User-Artist để dùng chung cho vòng lặp
full_user_items = item_user_matrix.T.tocsr()

for user_idx in range(sample_size):
    # SỬA LỖI: Hứng 2 mảng (ids, scores) và truyền đúng hàng dữ liệu [user_idx]
    ids, _ = loaded_model.recommend(
        userid=user_idx,
        user_items=full_user_items[user_idx],
        N=10,
        filter_already_liked_items=True
    )
    recommended_artists.update(ids)

# Công thức tính Coverage
coverage = (len(recommended_artists) / len(artist_map)) * 100

print(f"\n✓ KẾT QUẢ ĐÁNH GIÁ:")
print(f"  - Tổng số nghệ sĩ có trong hệ thống: {len(artist_map):,}")
print(f"  - Số nghệ sĩ được máy gợi ý (ít nhất 1 lần): {len(recommended_artists):,}")
print(f"  - Chỉ số Coverage: {coverage:.2f}%")

print("\n" + "=" * 60)
print("✓ CHÚC MỪNG: BASELINE MODEL ĐÃ HOÀN TẤT VÀ SẴN SÀNG BÀN GIAO!")
print("=" * 60)

ĐÁNH GIÁ CHẤT LƯỢNG BASELINE MODEL
Đang tính toán Coverage dựa trên 200 người dùng mẫu...

✓ KẾT QUẢ ĐÁNH GIÁ:
  - Tổng số nghệ sĩ có trong hệ thống: 17,632
  - Số nghệ sĩ được máy gợi ý (ít nhất 1 lần): 567
  - Chỉ số Coverage: 3.22%

✓ CHÚC MỪNG: BASELINE MODEL ĐÃ HOÀN TẤT VÀ SẴN SÀNG BÀN GIAO!
